# 🔮 De Prompts a Agentes: Sistemas Inteligentes con Python

**Workshop PyCon 2026 - Nivel Principiante**

En este taller aprenderemos a transformar prompts simples en agentes funcionales que pueden:
- ✅ Recibir preguntas en lenguaje natural
- ✅ Razonar y generar código
- ✅ Ejecutar acciones reales (análisis de datos)
- ✅ Explicar resultados de forma estructurada
- ✅ Recordar contexto y mejorar consistencia

**Tiempo total**: 2 horas

**Requisitos**: Python 3.10+, pandas y Ollama con `gemma2:2b`.

> La carpeta `agente/` contiene el proyecto práctico reutilizable. El notebook es la guía conceptual y de experimentación.

## Parte 1: ¿Qué es un Agente de IA?

### Analogía simple

**Un prompt es como dar una instrucción:**
- "Resume este texto" → Te da un resumen (pero podría inventar detalles)
- "Traduce esto al inglés" → Te da una traducción (pero podría tener errores)

**Un agente es como tener un asistente que piensa:**
- Recibe la pregunta
- Decide QUÉ acción tomar
- LA EJECUTA DE VERDAD (no solo inventa)
- Explica el resultado con evidencia real

### Los 4 componentes de un agente

| Componente | Descripción | Ejemplo |
|---|---|---|
| 👁 **Percepción** | Recibe la pregunta y el contexto | "¿Cuál fue la mayor venta?" + dataset CSV |
| 🧠 **Razonamiento** | Un LLM decide qué código escribir | Genera: `df.max()` |
| 🛠 **Herramienta** | Python ejecuta el código real | Pandas calcula: 5000 |
| 💬 **Respuesta** | El LLM convierte resultado a texto | "La mayor venta fue $5,000" (en JSON) |

### Diferencia clave

```
PROMPT SIMPLE:
  Entrada: "¿Cuál es el promedio de ventas?"
  LLM → (inventa un número) → "El promedio es $3,500"
  ❌ Podría ser falso

AGENTE:
  Entrada: "¿Cuál es el promedio de ventas?" + df
  LLM → Genera código: "df['ventas'].mean()"
  Python → Ejecuta → Resultado: 3245.67
  LLM → Explica → "El promedio es $3,245.67"
  ✅ Es un hecho verificable
```

## Parte 2: Configuración Inicial

### Instalación de dependencias

In [ ]:
# Ejecuta esto si no tienes las librerías instaladas
# !pip install pandas ollama

import pandas as pd
import json
from datetime import datetime
import os
from typing import Optional

from agente import cargar_ventas

print("✅ Librerías importadas correctamente")

### Crear dataset de ejemplo

In [ ]:
# Cargamos un dataset de ejemplo desde la carpeta agente

df = cargar_ventas()

print("📊 Dataset de ventas:")
print(df)
print(f"\n📈 Información del dataset:")
print(f"   - Filas: {len(df)}")
print(f"   - Columnas: {', '.join(df.columns)}")
print(f"   - Tipos: {dict(df.dtypes)}")

## Parte 3: De Prompts Simples a Estructurados

### Paso 1: El prompt más simple (❌ Problema)

In [ ]:
# ❌ PROMPT SIMPLE SIN CONTEXTO
prompt_basico = "¿Cuál es el promedio de ventas?"

print("❌ Problema del prompt básico:")
print(f"   Entrada: {prompt_basico}")
print(f"\n   Sin contexto, el LLM no sabe:")
print(f"   - ¿De qué datos hablas?")
print(f"   - ¿Qué columna es 'ventas'?")
print(f"   - ¿Qué formato quieres para la respuesta?")
print(f"   - ¿Debería ser una respuesta de texto o datos estructurados?")
print(f"\n   Resultado: El modelo INVENTA un número")

### Paso 2: Prompt con contexto (✅ Mejor)

In [ ]:
# ✅ PROMPT CON CONTEXTO
contexto_dataset = f"""
Tienes un dataset con las siguientes columnas:
- fecha (date): Fecha de la transacción
- producto (str): Producto vendido (A, B, C)
- region (str): Región de venta (Norte, Sur, Centro, Oriente)
- vendedor (str): Nombre del vendedor
- ventas (int): Monto de la venta en pesos
- cantidad (int): Cantidad de unidades

Total de registros: {len(df)}
"""

prompt_con_contexto = f"""{contexto_dataset}

Pregunta: ¿Cuál es el promedio de ventas?

Responde con un número exacto basado en el dataset.
"""

print("✅ Prompt mejorado con contexto:")
print(prompt_con_contexto)
print(f"\nAhora el modelo SABE:")
print(f"   - Qué columnas existen")
print(f"   - Qué significa 'ventas'")
print(f"   - Cuántos datos hay")
print(f"   - Que debe ser exacto")

### Paso 3: Prompt con restricciones y formato JSON

In [ ]:
# ✅ PROMPT CON CONTEXTO + RESTRICCIONES + FORMATO JSON
system_prompt_analizador = """
Eres un experto en análisis de datos.

Tu tarea es analizar un dataset y generar código Python usando pandas.

RESTRICCIONES:
1. Solo usa pandas (pd) y la variable 'df'
2. Genera ÚNICAMENTE código Python, sin explicaciones
3. El código debe ser una sola línea o un bloque breve
4. NO incluyas 'print()' en el código
5. La última línea debe ser el resultado (ej: df.mean())

FORMATO:
Responde ÚNICAMENTE con el código Python. Por ejemplo:
df['ventas'].mean()

o

df.groupby('region')['ventas'].sum().sort_values(ascending=False)
"""

user_prompt_analizador = f"""{contexto_dataset}

Genera código Python para responder: ¿Cuál es el promedio de ventas?
"""

print("✅ Prompt estructurado para generar código:")
print(f"\nSYSTEM PROMPT (contexto global):")
print(system_prompt_analizador)
print(f"\nUSER PROMPT (pregunta específica):")
print(user_prompt_analizador)

## Parte 4: Ejecutar Código Generado (Con Seguridad)

### El ciclo completo

In [ ]:
def ejecutar_codigo_pandas(codigo: str, df: pd.DataFrame, max_seguro: bool = True) -> tuple[bool, any, str]:
    """
    Ejecuta código pandas de forma controlada.
    
    Args:
        codigo: Código Python a ejecutar
        df: DataFrame a analizar
        max_seguro: Si True, rechaza palabras peligrosas
    
    Returns:
        (éxito, resultado, mensaje)
    """
    
    # Palabras bloqueadas para seguridad básica
    palabras_peligrosas = ['import', 'open', 'exec', 'eval', 'os', 'sys', 
                          'subprocess', 'requests', '__import__']
    
    # Validar código
    if max_seguro:
        for palabra in palabras_peligrosas:
            if palabra in codigo.lower():
                return False, None, f"❌ Código bloqueado: contiene '{palabra}'"
    
    try:
        # Crear namespace seguro
        namespace = {
            'df': df.copy(),
            'pd': pd,
            '__builtins__': {'len': len, 'sum': sum, 'min': min, 'max': max, 'list': list}
        }
        
        # Ejecutar código
        exec(codigo, namespace)
        
        # Extraer resultado (última expresión)
        resultado = eval(codigo, namespace)
        
        return True, resultado, "✅ Código ejecutado exitosamente"
    
    except Exception as e:
        return False, None, f"❌ Error al ejecutar: {str(e)}"


# Prueba con un código seguro
codigo_ejemplo = "df['ventas'].mean()"

print(f"📝 Ejecutando: {codigo_ejemplo}")
exito, resultado, mensaje = ejecutar_codigo_pandas(codigo_ejemplo, df)

print(f"{mensaje}")
if exito:
    print(f"📊 Resultado: {resultado}")
    print(f"   Tipo: {type(resultado).__name__}")

## Parte 5: Estructurar Respuestas en JSON

### ¿Por qué JSON?
- ✅ Máquinas pueden parsear fácilmente
- ✅ Formato consistente
- ✅ Fácil de almacenar o pasar a otro sistema
- ✅ Mejor que texto libre (el modelo podría hacer cualquier cosa)

In [ ]:
def crear_respuesta_json(valor: any, explicacion: str, insight: str = "") -> dict:
    """
    Estructura una respuesta en formato JSON estándar.
    """
    return {
        "valor": str(valor),
        "explicacion": explicacion,
        "insight": insight if insight else "N/A"
    }


# Ejemplo: después de ejecutar código pandas
exito, resultado, _ = ejecutar_codigo_pandas("df['ventas'].mean()", df)

if exito:
    respuesta = crear_respuesta_json(
        valor=resultado,
        explicacion="El promedio de todas las ventas registradas en el dataset.",
        insight="Las ventas varían entre $900 y $2,200, con un promedio equilibrado."
    )
    
    print("\n📋 Respuesta estructurada (JSON):")
    print(json.dumps(respuesta, ensure_ascii=False, indent=2))

## Parte 6: Verificar Consistencia

### El problema
Un LLM podría:
- Calcular bien el código
- Pero explicar MAL el resultado

### La solución
Verificar que los números que menciona en su explicación **coincidan** con el resultado real

In [ ]:
import re

def verificar_consistencia(valor_calculado: any, explicacion_texto: str) -> tuple[bool, str]:
    """
    Verifica si el valor calculado aparece en la explicación del modelo.
    """
    # Una Serie tiene varios nombres y valores; un escalar tiene uno.
    if isinstance(valor_calculado, pd.Series):
        candidatos = [str(x) for x in valor_calculado.index]
        candidatos += [str(x) for x in valor_calculado.tolist()]
    else:
        candidatos = [str(valor_calculado)]
        try:
            candidatos.append(str(round(float(valor_calculado), 2)))
        except (TypeError, ValueError):
            pass

    texto_normalizado = explicacion_texto.replace(",", "")
    encontrado = any(candidato in texto_normalizado for candidato in candidatos)
    
    if encontrado:
        return True, "✅ Explicación consistente con el resultado"
    else:
        return False, "⚠️ El valor en la explicación no coincide con el cálculo"


# Ejemplos
print("Prueba 1: Respuesta consistente")
ok1, msg1 = verificar_consistencia(
    valor_calculado=1455.0,
    explicacion_texto="El promedio de ventas es 1455 pesos"
)
print(f"{msg1}\n")

print("Prueba 2: Respuesta INCONSISTENTE")
ok2, msg2 = verificar_consistencia(
    valor_calculado=1455.0,
    explicacion_texto="El promedio de ventas es 2000 pesos"
)
print(f"{msg2}")

## Parte 7: Construir un Agente Simple

Ahora integramos todo en un **agente funcional**

In [ ]:
class AgenteAnalista:
    """
    Agente simple que:
    1. Recibe una pregunta
    2. Genera código pandas
    3. Lo ejecuta
    4. Explica el resultado
    5. Verifica consistencia
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.schema = self._extraer_schema()
        self.historial = []  # Para memoria
    
    def _extraer_schema(self) -> str:
        """Extrae información del dataset"""
        tipos = {col: str(dtype) for col, dtype in self.df.dtypes.items()}
        schema = f"Columnas: {', '.join(self.df.columns)} | Tipos: {tipos}"
        return schema
    
    def procesar_pregunta(self, pregunta: str, codigo_simulado: str = None) -> dict:
        """
        Procesa una pregunta y devuelve respuesta estructurada.
        
        En un agente real, aquí usarías un LLM para generar el código.
        Para esta demostración, simulamos con código pre-escrito.
        """
        
        print(f"\n{'='*60}")
        print(f"❓ Pregunta: {pregunta}")
        print(f"{'='*60}")
        
        # En un agente real, aquí llamarías al LLM para generar código
        if codigo_simulado is None:
            return {"error": "Se requiere código para esta demostración"}
        
        # Paso 1: Mostrar código generado
        print(f"\n📝 Código generado:")
        print(f"   {codigo_simulado}")
        
        # Paso 2: Ejecutar código
        print(f"\n🛠 Ejecutando código...")
        exito, resultado, mensaje = ejecutar_codigo_pandas(codigo_simulado, self.df)
        print(f"   {mensaje}")
        
        if not exito:
            return {"error": mensaje}
        
        # Paso 3: Generar explicación (simular LLM)
        print(f"\n💬 Explicación generada (simulada):")
        explicacion = self._generar_explicacion(pregunta, resultado)
        print(f"   {explicacion}")
        
        # Paso 4: Estructurar respuesta
        respuesta = crear_respuesta_json(
            valor=resultado,
            explicacion=explicacion,
            insight=f"Resultado calculado con {len(self.df)} registros en el dataset"
        )
        
        # Paso 5: Verificar consistencia
        print(f"\n🔍 Verificando consistencia...")
        ok, verificacion = verificar_consistencia(resultado, explicacion)
        print(f"   {verificacion}")
        
        respuesta["consistencia"] = ok
        
        # Guardar en historial
        self.historial.append({
            "pregunta": pregunta,
            "respuesta": respuesta
        })
        
        return respuesta
    
    def _generar_explicacion(self, pregunta: str, resultado: any) -> str:
        """Simula una explicación; el proyecto final usa Ollama."""
        
        # Aquí iría una llamada real a un LLM
        # Por ahora, generamos explicaciones básicas
        
        if isinstance(resultado, (int, float)):
            return f"El resultado es {resultado:.2f}."
        else:
            return f"Resultado: {resultado}"
    
    def mostrar_respuesta_final(self, respuesta: dict):
        """Muestra la respuesta de forma legible"""
        print(f"\n📋 Respuesta Final (JSON):")
        print(json.dumps(respuesta, ensure_ascii=False, indent=2))


# Crear agente
agente = AgenteAnalista(df)

# Procesar una pregunta
respuesta = agente.procesar_pregunta(
    pregunta="¿Cuál es el promedio de ventas?",
    codigo_simulado="df['ventas'].mean()"
)

agente.mostrar_respuesta_final(respuesta)

## Parte 8: Agregar Memoria

Un agente inteligente recuerda el contexto anterior

In [ ]:
class AgenteConMemoria(AgenteAnalista):
    """
    Extiende el agente simple con memoria.
    
    Esto permite:
    - Preguntas de seguimiento ("¿Y cuál quedó segundo?")
    - Contexto acumulativo
    - Mejor razonamiento
    """
    
    def __init__(self, df: pd.DataFrame, max_memoria: int = 3):
        super().__init__(df)
        self.max_memoria = max_memoria
        self.conversacion = []
    
    def construir_contexto_memoria(self) -> str:
        """Crea un texto con el historial reciente para usar en prompts"""
        
        if not self.conversacion:
            return "(Sin historial previo)"
        
        contexto = "Historial reciente:\n"
        for i, turno in enumerate(self.conversacion[-self.max_memoria:], 1):
            contexto += f"{i}. P: {turno['pregunta']}\n"
            contexto += f"   R: {turno['respuesta'].get('explicacion', 'N/A')}\n"
        
        return contexto
    
    def procesar_pregunta_con_memoria(self, pregunta: str, codigo_simulado: str = None) -> dict:
        """Procesa pregunta considerando el historial"""
        
        # Mostrar contexto de memoria
        print(f"\n{'='*60}")
        print(f"❓ Nueva pregunta: {pregunta}")
        print(f"{'='*60}")
        print(f"\n🧠 Contexto de memoria:")
        print(self.construir_contexto_memoria())
        
        # Procesar normalmente
        respuesta = self.procesar_pregunta(pregunta, codigo_simulado)
        
        # Guardar en conversación
        self.conversacion.append({
            "pregunta": pregunta,
            "respuesta": respuesta,
            "timestamp": datetime.now().isoformat()
        })
        
        return respuesta


# Crear agente con memoria
agente_memoria = AgenteConMemoria(df)

# Primera pregunta
print("\n🔵 TURNO 1: Primera pregunta")
resp1 = agente_memoria.procesar_pregunta_con_memoria(
    pregunta="¿Cuál región tuvo más ventas?",
    codigo_simulado="df.groupby('region')['ventas'].sum().sort_values(ascending=False)"
)

# Segunda pregunta de seguimiento
print("\n🔵 TURNO 2: Pregunta de seguimiento")
resp2 = agente_memoria.procesar_pregunta_con_memoria(
    pregunta="¿Y cuál quedó en segundo lugar?",
    codigo_simulado="df.groupby('region')['ventas'].sum().sort_values(ascending=False).head(2)"
)

## Parte 9: Ejercicios Prácticos

### Ejercicio 1: Entender componentes

In [ ]:
# EJERCICIO 1: Identifica los 4 componentes en este flujo

print("""\n📚 EJERCICIO 1: Identificar componentes del agente

Dado este flujo:

1. Usuario pregunta: "¿Cuántas ventas hubo en la región Norte?"

2. LLM genera:
   df[df['region'] == 'Norte']['ventas'].sum()

3. Python ejecuta → Resultado: 4700

4. LLM explica:
   "La región Norte registró 4,700 en ventas totales,
    lo que la hace líder en el dataset."

Identifica:
  A) ¿Cuál es el componente de PERCEPCIÓN?
  B) ¿Cuál es el componente de RAZONAMIENTO?
  C) ¿Cuál es el componente de HERRAMIENTA?
  D) ¿Cuál es el componente de RESPUESTA?

Respuestas en: respuestas/ejercicio_1_solucion.md
""")

### Ejercicio 2: Escribir prompts estructurados

In [ ]:
print("""\n📚 EJERCICIO 2: Mejorar un prompt

Prompt original (MALO):
  "Analiza el dataset"

Tu tarea:
  - Mejora el prompt agregando CONTEXTO
  - Agrega RESTRICCIONES
  - Define un FORMATO específico para la respuesta

Pista: Usa los prompts que vimos en la Parte 3

Tu respuesta va en: respuestas/ejercicio_2_solucion.md
""")

### Ejercicio 3: Ejecutar código pandas

In [ ]:
print("""\n📚 EJERCICIO 3: Ejecutar y entender código pandas

Pregunta: ¿Cuál fue el vendedor con menos ventas totales?

Tu tarea:
  1. Escribe código pandas para responder
  2. Ejecútalo en la celda siguiente
  3. Verifica que sea correcto

Hint: Usa groupby + sum + sort_values

""")

# Aquí va tu código
# TU SOLUCIÓN:
# codigo = "..."
# exito, resultado, msg = ejecutar_codigo_pandas(codigo, df)
# print(resultado)

## Parte 10: Conectar el agente con Ollama

Ollama corre el modelo localmente. La función se define sin hacer una llamada; úsala después de ejecutar `ollama pull gemma2:2b`.

In [ ]:
# Definir esta función no consulta el modelo todavía.
from ollama import chat

ESQUEMA = {
    "type": "object",
    "properties": {
        "operacion": {"type": "string", "enum": ["promedio", "suma", "ranking"]},
        "columna": {"type": "string", "enum": ["ventas", "cantidad"]},
        "agrupar_por": {"type": ["string", "null"]},
        "orden": {"type": "string", "enum": ["asc", "desc"]},
        "limite": {"type": "integer"}
    },
    "required": ["operacion", "columna", "agrupar_por", "orden", "limite"]
}

def generar_decision_con_ollama(pregunta: str, contexto: str = "") -> dict:
    prompt = f"Contexto: {contexto}\nPregunta: {pregunta}\nGenera la decisión JSON."
    respuesta = chat(
        model="gemma2:2b",
        messages=[{"role": "user", "content": prompt}],
        format=ESQUEMA,
        options={"temperature": 0}
    )
    return json.loads(respuesta.message.content)

print("✅ Integración con Ollama definida. El proyecto ejecutable está en agente/")

## Parte 11: Resumen de Conceptos

### Lo que aprendimos

| Concepto | Explicación |
|---|---|
| **Prompts simples** | Generan texto pero pueden inventar |
| **Prompts estructurados** | Con contexto, restricciones y formato |
| **Agentes** | Sistemas que perciben, razonan, actúan y responden |
| **Ejecución segura** | Validar código antes de ejecutarlo |
| **JSON estructurado** | Respuestas máquina-legibles |
| **Verificación** | Comparar resultado con explicación |
| **Memoria** | Recordar contexto para mejores decisiones |

### Ciclo completo de un agente

```
Usuario → Pregunta
         ↓
LLM ← Razona qué código escribir ← Contexto + Restricciones
         ↓
Código generado → Validar → Ejecutar
         ↓
Resultado real ← Pandas/Python
         ↓
LLM ← Explicar resultado en lenguaje natural
         ↓
JSON Estructurado ← Verificar Consistencia
         ↓
Respuesta al usuario
         ↓
Guardar en Memoria ← Para próximas preguntas
```

### Próximos pasos

1. **Mejora el agente** con más validaciones
2. **Prueba el agente real** con Ollama local
3. **Añade más herramientas** (API calls, web scraping, etc.)
4. **Deploy a producción** (Flask API, Discord bot, etc.)

El proyecto completo y sus ejercicios están en: `agente/`

## Parte 12: Preguntas Frecuentes

### ¿Necesito un LLM para que funcione?
Sí. En este taller usamos únicamente **Ollama**, que corre localmente y no requiere API key.

### ¿Es seguro ejecutar código del LLM?
No automáticamente. Necesitas:
- Validar el código
- Usar un namespace controlado
- En producción: contenedores, VMs, o servicios como E2B

### ¿Qué pasa si el modelo genera código incorrecto?
Tres niveles de protección:
1. Validación: rechaza palabras peligrosas
2. Try/catch: captura errores de ejecución
3. Verificación: compara resultado con explicación

### ¿Cómo preparo Ollama?
1. Instala Ollama desde https://ollama.com
2. `ollama pull gemma2:2b`
3. `ollama serve` en una terminal
4. El agente se conecta automáticamente

Ver: `setup_ollama.md`

## Conclusión

🎓 **Conclusión del taller**

Comenzamos con prompts simples que inventan respuestas.

Evolucionamos a agentes inteligentes que:
- ✅ Razonan qué hacer
- ✅ Ejecutan acciones reales
- ✅ Verifican su propio trabajo
- ✅ Mejoran con memoria

**El verdadero poder de la IA no está en generar texto.**
**Está en construir sistemas que razonan y actúan.**

¡Ahora tienes las herramientas para hacerlo!

---

📁 **Recursos en la carpeta `respuestas/`:**
- Soluciones de ejercicios
- Código completo con LLM real
- Guías de deployment
- Ejemplos adicionales

📁 **Siguiente paso:** abre `agente/README.md`, ejecuta tu agente y completa los tres ejercicios.